# Clase 2 — Backpropagation con el problema XOR

## Objetivos de aprendizaje
- Entender las 5 etapas de backpropagation: forward pass, cálculo del error, backward pass, actualización de pesos, repetición por épocas.
- Implementar backpropagation con NumPy puro para entrenar una red que resuelva XOR.
- Observar cómo el error medio disminuye a lo largo de las épocas de entrenamiento.
- Entender por qué XOR (al no ser linealmente separable) es el ejemplo clásico para demostrar que una red con capa oculta aprende relaciones no lineales.

## Qué vamos a construir
Vamos a entrenar, desde cero, una red de 2 entradas, 2 nodos ocultos y 1 salida para que aprenda la función lógica XOR. Para que la demo se ejecute rápido en clase, vamos a usar menos épocas y una tasa de aprendizaje más alta que en el material original (que usa 180.000 épocas con `lr=0.1`) — vas a ver el mismo patrón de convergencia, solo que más rápido.

## Repaso teórico rápido: las 5 etapas de backpropagation

1. **Forward pass**: `X → w1 → sigmoide → a1 → w2 → sigmoide → a2` (la predicción).
2. **Cálculo del error**: `error = d - a2` (valor esperado menos predicción).
3. **Backward pass**: se calcula `dz2` combinando el error con la derivada de la sigmoide en
   la salida; se reparte hacia la capa oculta multiplicando por `w2.T` para obtener `da1`,
   y se combina de nuevo con la derivada de la sigmoide para obtener `dz1`.
4. **Actualización**: `w2 += lr*dz2·a1.T`, `b2 += lr*sum(dz2)`, `w1 += lr*dz1·X.T`,
   `b1 += lr*sum(dz1)`.
5. **Repetir** por muchas épocas, hasta que el error medio sea pequeño.

XOR no se puede separar con una sola línea recta, así que si la red aprende XOR es porque
realmente está combinando información de forma no lineal gracias a la capa oculta.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(1)  # reproducible en clase

# tabla de verdad de XOR: cada columna de X es un ejemplo (x1, x2)
X = np.array([[0, 0, 1, 1],
              [0, 1, 0, 1]])
d = np.array([[0, 1, 1, 0]])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def initialize_network_parameters():
    inputSize = 2
    hiddenSize = 2
    outputSize = 1
    # NOTA: en el material original se usa lr=0.1 y epochs=180000.
    # Aquí usamos una tasa de aprendizaje más alta y menos épocas
    # para que la celda se ejecute rápido durante la clase en vivo.
    lr = 0.5
    epochs = 20000

    w1 = np.random.uniform(size=(hiddenSize, inputSize), low=-1, high=1)
    b1 = np.random.uniform(size=(hiddenSize, 1), low=-1, high=1)
    w2 = np.random.uniform(size=(outputSize, hiddenSize), low=-1, high=1)
    b2 = np.random.uniform(size=(outputSize, 1), low=-1, high=1)

    return w1, b1, w2, b2, lr, epochs

w1, b1, w2, b2, lr, epochs = initialize_network_parameters()
error_list = []

for epoch in range(epochs):
    # 1. Forward pass
    z1 = np.dot(w1, X) + b1
    a1 = sigmoid(z1)
    z2 = np.dot(w2, a1) + b2
    a2 = sigmoid(z2)

    # 2. Cálculo del error
    error = d - a2

    # 3. Backward pass
    dz2 = error * a2 * (1 - a2)
    dw2 = np.dot(dz2, a1.T)
    db2 = np.sum(dz2, axis=1, keepdims=True)

    da1 = np.dot(w2.T, dz2)
    dz1 = da1 * a1 * (1 - a1)
    dw1 = np.dot(dz1, X.T)
    db1 = np.sum(dz1, axis=1, keepdims=True)

    # 4. Actualización de pesos y sesgos
    w2 += lr * dw2
    b2 += lr * db2
    w1 += lr * dw1
    b1 += lr * db1

    error_list.append(np.mean(np.abs(error)))

    if epoch % 2000 == 0:
        print('Época {}: error medio = {:.5f}'.format(epoch, np.mean(np.abs(error))))

In [ ]:
# 5. Test final: ¿aprendió la red a resolver XOR?
z1 = np.dot(w1, X) + b1
a1 = sigmoid(z1)
z2 = np.dot(w2, a1) + b2
a2 = sigmoid(z2)

print('Predicciones finales:', np.round(a2, 5))
print('Valores esperados:   ', d)

plt.plot(error_list)
plt.title('Error medio a lo largo de las épocas (XOR)')
plt.xlabel('Época')
plt.ylabel('Error medio absoluto')
plt.grid(True)
plt.show()

## Resumen de la clase

- Backpropagation repite 5 etapas: forward pass, error, backward pass, actualización, y
  vuelta a empezar por muchas épocas.
- El error medio de XOR debería empezar cerca de 0.5 (adivinando al azar) y bajar de forma
  constante conforme avanzan las épocas.
- Usamos una configuración de hiperparámetros distinta a la del material original
  (`lr=0.5`, `epochs=20000` en vez de `lr=0.1`, `epochs=180000`) solo para que la demo sea
  rápida en clase — el patrón de convergencia es el mismo, solo que con menos precisión
  final. En el capítulo del manual puedes ver la configuración original completa.
- XOR es el ejemplo clásico porque demuestra que la capa oculta permite aprender una
  relación que no es linealmente separable.

**Para seguir profundizando:**
- Capítulo del manual: [`manual/04_backpropagation.md`](../manual/04_backpropagation.md)
- Notebook de práctica de esta clase: [`notebooks_practica/clase02_practica_backpropagation.ipynb`](../notebooks_practica/clase02_practica_backpropagation.ipynb)